<a href="https://colab.research.google.com/github/sharnitha567/Machine-Learning/blob/main/Experiment_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [35]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    classification_report
)

In [36]:
df = pd.read_csv("/content/student_performance_updated_1000.csv")

print("Dataset loaded successfully!")
print("Dataset Shape:", df.shape)



Dataset loaded successfully!
Dataset Shape: (1000, 12)


In [37]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("%", "percent", regex=False)
)

In [38]:
df.drop(
    columns=["studentid", "name"],
    errors="ignore",
    inplace=True
)

In [39]:
df.replace([np.inf, -np.inf], np.nan, inplace=True)

In [40]:
# Attendance should be between 0 and 100
for column in ["attendancerate", "attendance_percent"]:
    if column in df.columns:
        df.loc[
            (df[column] < 0) | (df[column] > 100),
            column
        ] = np.nan

# Previous grade and final grade should be 0-100
for column in ["previousgrade", "finalgrade"]:
    if column in df.columns:
        df.loc[
            (df[column] < 0) | (df[column] > 100),
            column
        ] = np.nan

# Study hours cannot be negative
for column in ["studyhoursperweek", "study_hours"]:
    if column in df.columns:
        df.loc[
            df[column] < 0,
            column
        ] = np.nan

In [41]:
df.drop_duplicates(inplace=True)

In [42]:

def performance_class(grade):

    if grade < 50:
        return "Fail"

    elif grade < 75:
        return "Average"

    else:
        return "Good"


df["performance"] = df["finalgrade"].apply(
    performance_class
)

print("\nPerformance Classes:")
print(df["performance"].value_counts())


Performance Classes:
performance
Good       707
Average    293
Name: count, dtype: int64


In [43]:
# FinalGrade is used to create the class.
# It must NOT be used as an input feature.

df.drop(
    columns=["finalgrade"],
    inplace=True
)

In [44]:
X = df.drop(columns=["performance"])
y = df["performance"]


In [45]:
X = pd.get_dummies(
    X,
    drop_first=True,
    dtype=int
)


In [46]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("\nTraining data:", X_train.shape)
print("Testing data :", X_test.shape)



Training data: (800, 10)
Testing data : (200, 10)


In [47]:
train_medians = X_train.median()

X_train = X_train.fillna(train_medians)
X_test = X_test.fillna(train_medians)

In [48]:
dt_model = DecisionTreeClassifier(
    random_state=42,
    max_depth=5
)

dt_model.fit(X_train, y_train)

DecisionTreeClassifier(max_depth=5, random_state=42)

In [49]:
y_pred = dt_model.predict(X_test)

In [50]:
cm = confusion_matrix(
    y_test,
    y_pred,
    labels=["Fail", "Average", "Good"]
)

print("\n===================================")
print("CONFUSION MATRIX")
print("===================================")

print(cm)




CONFUSION MATRIX
[[  0   0   0]
 [  0  10  49]
 [  0  12 129]]


In [51]:
accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\n===================================")
print("ACCURACY")
print("===================================")

print("Accuracy:", accuracy)
print("Accuracy Percentage:", accuracy * 100, "%")


ACCURACY
Accuracy: 0.695
Accuracy Percentage: 69.5 %


In [52]:
print("\n===================================")
print("CLASSIFICATION REPORT")
print("===================================")

print(
    classification_report(
        y_test,
        y_pred,
        labels=["Fail", "Average", "Good"],
        zero_division=0
    )
)



CLASSIFICATION REPORT
              precision    recall  f1-score   support

        Fail       0.00      0.00      0.00         0
     Average       0.45      0.17      0.25        59
        Good       0.72      0.91      0.81       141

    accuracy                           0.69       200
   macro avg       0.39      0.36      0.35       200
weighted avg       0.65      0.69      0.64       200



In [53]:
print("\n===================================")
print("SAMPLE PREDICTIONS")
print("===================================")

sample_output = pd.DataFrame({
    "Actual": y_test.iloc[:10].values,
    "Predicted": y_pred[:10]
})

print(sample_output)


SAMPLE PREDICTIONS
    Actual Predicted
0     Good      Good
1     Good   Average
2  Average      Good
3     Good      Good
4     Good      Good
5  Average      Good
6     Good   Average
7  Average      Good
8     Good   Average
9     Good   Average
